# Rubin RSP access examples

This notebook demonstrates the collection-aware helpers in `CollectionContext` and `target_selection.rsp_access`.
They provide a small, explicit interface for Rubin/RSP tables, coadd property maps, visit coverage, coadd coverage, and coadd cutouts.

The examples are intentionally not executed automatically because they require an RSP kernel with TAP and Butler access.


## 1. Imports and collection

Every query receives a collection name. Collection-specific table names, Butler dataset types, and capabilities are resolved from the release profile.


In [ ]:
from pathlib import Path
import pandas as pd

from target_selection import open_collection

collection = open_collection("DP2")
BANDS = tuple("ugrizy")


## 2. Discover what a collection provides

Use these functions before writing a query. They return the configured table aliases, property-map aliases, and image/product families for the selected collection.


In [ ]:
collection.list_tables()

In [ ]:
collection.list_property_maps()

In [ ]:
collection.list_images()

In [ ]:
capabilities = collection.describe()
print(capabilities["notes"])

For DP2, the important distinction is that `visit_coverage` is a table-level product. It does not mean that a calibrated single-visit image (`calexp`) is available for download. Coadd cutouts are the supported image product in this API.


## 3. Define target positions

The position-based helpers use a small canonical table with `target_id`, `ra_deg`, and `dec_deg`. `RA`/`DEC` are also accepted.


In [ ]:
targets = pd.DataFrame({
    "target_id": ["example-1", "example-2"],
    "ra_deg": [10.0, 10.1],
    "dec_deg": [-20.0, -20.1],
})
targets

## 4. Query a TAP table

Use `filters` for simple equality or `IN` conditions, and `where` for compound or spatial ADQL expressions.


In [ ]:
visit_rows = collection.get_table(
    "VisitDetector",
    collection=collection,
    columns=("visitId", "band", "detector", "seeing"),
    filters={"band": ["r", "i"]},
    limit=10,
)
visit_rows.head()

In [ ]:
dia_objects = collection.get_table(
    "dp2.DiaObject",
    collection=collection,
    columns=("diaObjectId", "ra", "dec"),
    where="ra BETWEEN 10 AND 11 AND dec BETWEEN -21 AND -19",
    limit=10,
)
dia_objects.head()

## 5. Query individual-visit coverage

This performs the spatial visit/detector matching and returns target-level coverage by band. It is metadata, not an image download.


In [ ]:
visit_coverage = collection.get_visit_coverage(
    targets,
    bands=BANDS,
    tile_nside=64,
    max_workers=4,
    cache_dir=Path("outputs/rsp_cache/visit_coverage"),
)
visit_coverage.head()

## 6. Sample coadd property maps

Properties are sampled at the target coordinates from consolidated coadd property maps. They are not measured from the cutout pixels.


In [ ]:
properties = collection.get_property_map_values(
    targets,
    properties=("maglim", "psf_size", "exposure_time", "sky_noise"),
    bands=("g", "r", "i"),
    partial_reads=True,
    cache_path=Path("outputs/rsp_cache/example_properties.csv"),
)
properties.head()

collection.list_property_maps()

## 7. Find coadds and generate cutouts

First add direct Butler coadd coverage. Then generate 20-arcsecond cutouts only for rows with coadd coverage.


In [ ]:
coadd_coverage = collection.get_coadd_coverage(
    visit_coverage,
    cache_path=Path("outputs/rsp_cache/coadd_coverage.csv"),
    bands=BANDS,
)
coadd_coverage.head()

In [ ]:
manifest = collection.get_coadd_cutouts(
    coadd_coverage,
    output_dir=Path("outputs/rsp_cutouts/pngs"),
    fits_dir=Path("outputs/rsp_cutouts/fits"),
    bands=BANDS,
    size_arcsec=20.0,
    category_column=None,
    add_rgb=False,
)
manifest.head()

## 8. Recommended pattern

Keep table queries, property-map sampling, coverage, and image extraction as separate operations. This makes the provenance explicit and avoids treating a coverage row, a property-map value, and a science image as if they were the same kind of data.
